# 🎯 Routing and Filtering — Search Smarter, Not Harder

## The Idea

When your knowledge base grows large, searching everything becomes:
- **Slow** — more docs to compare against
- **Noisy** — unrelated docs pollute results

**Routing** sends the query to the right *subset* of documents.  
**Filtering** removes documents that don't match certain *criteria* (date, category, author, etc.).

```
Without routing:
   Query: "Python syntax error"  →  search ALL 10,000 docs
   Result: programming docs + cooking docs + finance docs + ...

With routing:
   Query: "Python syntax error"  →  detect topic = 'programming'
                                 →  search only 1,200 programming docs
   Result: precise, fast, relevant
```

## What You'll Learn

1. Metadata filtering — filter by tags, date, category
2. Semantic routing — classify query, route to the right collection
3. Combined: route + filter + search

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

## 1. Documents With Metadata

In real RAG systems, every document has metadata attached — not just the text.

In [ ]:
# Documents with metadata
# Each doc has: text + category + source + year

docs = [
    {"id": 0, "text": "Gradient descent minimizes the loss by updating weights in the direction of the negative gradient.",
     "category": "ml_fundamentals", "source": "textbook", "year": 2021},

    {"id": 1, "text": "Transformers use self-attention to model relationships between all tokens in a sequence simultaneously.",
     "category": "deep_learning", "source": "paper", "year": 2017},

    {"id": 2, "text": "RAG retrieves relevant documents and passes them as context to the language model at inference time.",
     "category": "rag", "source": "blog", "year": 2023},

    {"id": 3, "text": "GPT-4 achieves strong performance on reasoning tasks by scaling both data and model parameters.",
     "category": "llm", "source": "paper", "year": 2023},

    {"id": 4, "text": "Vector databases like Chroma and Pinecone store embeddings and support fast nearest-neighbor search.",
     "category": "rag", "source": "docs", "year": 2023},

    {"id": 5, "text": "Backpropagation computes gradients by applying the chain rule from the output layer backward.",
     "category": "ml_fundamentals", "source": "textbook", "year": 2019},

    {"id": 6, "text": "Fine-tuning a pretrained LLM on domain-specific data significantly improves task performance.",
     "category": "llm", "source": "blog", "year": 2024},

    {"id": 7, "text": "Chunking strategy affects retrieval quality — overlapping chunks preserve context across boundaries.",
     "category": "rag", "source": "blog", "year": 2024},

    {"id": 8, "text": "Convolutional neural networks extract hierarchical features from images using learned filters.",
     "category": "deep_learning", "source": "textbook", "year": 2020},

    {"id": 9, "text": "Embedding models convert text to dense vectors. Recent models like E5 and BGE outperform BERT.",
     "category": "rag", "source": "paper", "year": 2024},
]

# Embed all document texts
texts = [d["text"] for d in docs]
embeddings = model.encode(texts)

print(f"Indexed {len(docs)} documents")
print(f"Categories: {set(d['category'] for d in docs)}")

## 2. Metadata Filtering

Filter documents by their metadata *before* doing similarity search.

In [ ]:
# Without filtering — search everything
query = "how does retrieval work in RAG?"

q_emb = model.encode(query)
all_scores = cosine_similarity([q_emb], embeddings)[0]
top_all = np.argsort(all_scores)[::-1][:4]

print("No filter — top 4 results from ALL docs:")
for idx in top_all:
    print(f"  [{all_scores[idx]:.3f}] [{docs[idx]['category']:15}] {docs[idx]['text'][:65]}...")

In [ ]:
# With filtering — only search docs in the 'rag' category

# Step 1: Filter docs by metadata
rag_docs = [d for d in docs if d["category"] == "rag"]
rag_indices = [d["id"] for d in rag_docs]
rag_embeddings = embeddings[rag_indices]  # Only their embeddings

# Step 2: Search only within the filtered subset
q_emb = model.encode(query)
filtered_scores = cosine_similarity([q_emb], rag_embeddings)[0]
top_filtered = np.argsort(filtered_scores)[::-1][:4]

print(f"Filtered to 'rag' category ({len(rag_docs)} docs) — top 4:")
for rank_idx in top_filtered:
    doc = rag_docs[rank_idx]
    score = filtered_scores[rank_idx]
    print(f"  [{score:.3f}] [{doc['category']:15}] {doc['text'][:65]}...")

In [ ]:
# Filter by multiple criteria: category AND year >= 2023

recent_rag_docs = [d for d in docs if d["category"] == "rag" and d["year"] >= 2023]

print(f"Docs matching category='rag' AND year>=2023: {len(recent_rag_docs)}\n")
for d in recent_rag_docs:
    print(f"  [{d['year']}] {d['text'][:70]}...")

## 3. Semantic Routing

Instead of hardcoding the category filter, **detect the category automatically** from the query text.

In [ ]:
# Define one representative description per category
# The router embeds these and the query, then picks the closest category

routes = {
    "ml_fundamentals": "gradient descent, backpropagation, loss functions, optimization, training basics",
    "deep_learning":   "neural networks, transformers, CNNs, attention mechanisms, architectures",
    "rag":             "retrieval augmented generation, vector search, embeddings, chunking, indexing",
    "llm":             "large language models, GPT, fine-tuning, prompting, RLHF, instruction tuning",
}

# Embed each route description
route_names = list(routes.keys())
route_embeddings = model.encode(list(routes.values()))

print("Routes defined:", route_names)

In [ ]:
# Router function: classify query into a category

def route_query(query):
    q_emb = model.encode(query)
    # Similarity between query and each route description
    scores = cosine_similarity([q_emb], route_embeddings)[0]
    best_idx = np.argmax(scores)
    return route_names[best_idx], scores[best_idx]


# Test routing with different queries
test_queries = [
    "how does RAG retrieve documents?",
    "what is gradient descent?",
    "how do I fine-tune GPT?",
    "how does self-attention work?",
]

print("Query routing results:\n")
for q in test_queries:
    category, confidence = route_query(q)
    print(f"  '{q}'")
    print(f"   → Route: {category} (confidence: {confidence:.3f})\n")

## 4. Full Pipeline: Route → Filter → Search

In [ ]:
# Combined pipeline

def routed_search(query, top_k=3):
    # Step 1: Detect which category this query belongs to
    category, confidence = route_query(query)
    print(f"Routed to: '{category}' (confidence: {confidence:.3f})")

    # Step 2: Filter docs to that category only
    filtered_docs = [d for d in docs if d["category"] == category]
    filtered_embs = np.array([embeddings[d["id"]] for d in filtered_docs])
    print(f"Searching {len(filtered_docs)}/{len(docs)} docs")

    # Step 3: Search within the filtered set
    q_emb = model.encode(query)
    scores = cosine_similarity([q_emb], filtered_embs)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = [(filtered_docs[i], float(scores[i])) for i in top_indices]
    return results


# Try it
for query in ["how does RAG chunking work?", "explain backpropagation"]:
    print(f"\nQuery: '{query}'")
    results = routed_search(query)
    print("Results:")
    for doc, score in results:
        print(f"  [{score:.3f}] {doc['text'][:65]}...")

## Key Takeaways 🎯

| Technique | What It Does | When to Use |
|---|---|---|
| **Metadata filter** | Exclude docs by tag/date/source | When you know the filter criteria upfront |
| **Semantic routing** | Auto-detect category from query | When category is implicit in the query |
| **Route + filter** | Classify then filter | Best of both — most production systems |

**Key rule:** The smaller the search space, the faster and more precise retrieval becomes.

---
**Section 3 complete!** Next: Section 4 — Generation.